In [1]:
#imports
from pathlib import Path
import subprocess
import sys
import pandas as pd
import json

# Data preprocessing

Builds a clean dataset from the raw upper-limb recordings. The raw files are **never modified**. Everything is written to `data/processed/`.

The dataset is a set of linked tables (see the cheat sheet in `explore_data.ipynb`):

## Vocabulary: session, task, trial, frame

Think 1 session as a visit to the physiotherapist:

| word | meaning | analogy | example |
|---|---|---|---|
| **session** | everything recorded between "start" and "stop" | **one visit** of one patient | `upper_limb_20260629_095022`: 29 June, 09:50, 13.7 min |
| **task** | one of the 8 exercises (numbered 1 to 8) | **one exercise** | task 1 = "Shoulder flexion with elbow extended" (raise the arm in front) |
| **trial** | one single movement: start position → movement → back to start | **one repetition** | the 1st arm raise of task 1 |
| **frame** | one snapshot of the body points (shoulder, elbow, wrist, ...) | **one photo** | ~17 per second, so a 4 s movement ≈ 70 frames |

The example session, as a tree:

```
SESSION upper_limb_20260629_095022              1 visit, 13.7 min, right arm
 ├─ TASK 1  Shoulder flexion with elbow extended     10 trials, ~4.3 s each
 │   ├─ TRIAL 1   one arm raise   → ~75 frames
 │   ├─ TRIAL 2   one arm raise   → ~75 frames
 │   └─ ...  up to TRIAL 10
 ├─ TASK 2  Shoulder abduction with elbow extended   10 trials, ~4.2 s each
 ├─ TASK 3  Draw a 5 segment trajectory              10 trials, ~9.8 s each
 ├─ TASK 4  Reaching different targets               30 trials, ~2.3 s each
                                                   = 75 trials in total
```

In the whole dataset: **79 usable sessions**, **8 tasks**, **3,013 trials** and about 450,000 frames inside those trials.

Two details:
- **`task_run`**: the operator sometimes starts the same task twice in one session (after a mistake, or for the other arm). The second run has `task_run = 2`, and its trials are numbered from 1 again.
- **The patient** is linked to the session, not to trials or frames: 1 patient → sessions → trials → frames.

For the analysis (e.g. PCA), **one trial = one movement = one data point**.

In [2]:
project_root = Path.cwd().parent
recordings_root = project_root / "data" / "upper_limb_recorded_data" / "upper_limb_asessment" / "recordings"  # raw data (read only)
processed_root = project_root / "data" / "processed"  # where the clean tables will be saved

# List of all sessions = every "upper_limb_*.events.jsonl" file, without its extension.
session_ids = sorted(p.name.removesuffix(".events.jsonl") for p in recordings_root.glob("upper_limb_*.events.jsonl"))
print(f"Upper-limb sessions found: {len(session_ids)}")


def read_events(session_id):
    """All events of one session, in file order. Returns [] for an empty file."""
    path = recordings_root / f"{session_id}.events.jsonl"
    with path.open(encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()] # One JSON object per line -> one Python dict per event. Blank lines are skipped.

Upper-limb sessions found: 112


## Step 1: sessions table (one row per session)

What we learned about the sessions:

- **`patient_id` is not in `session_start`** (it is always empty there). The operator types it during the session, which creates a `patient_id_set` event. The dominant arm works the same way (`dominant_arm_set`).
- Only sessions from 29 July onwards have a patient ID. Some IDs are clearly software tests (`test_*`, `tester`).
- 4 sessions have an **empty events file** (0 bytes).
- Some sessions have no trials at all (short tests or aborted starts).

Columns:

| column | meaning |
|---|---|
| `session_id` | file name, e.g. `upper_limb_20260910_133402` |
| `date`, `start_time` | from the file name |
| `patient_id` | last `patient_id_set` value in the session (empty = never entered) |
| `patient_ids_entered` | all IDs typed during the session, to spot sessions where the ID changed |
| `dominant_arm` | all `dominant_arm_set` values, e.g. `right` or `left, right` |
| `looks_like_test` | patient ID starts with `test` (includes `tester`) |
| `n_events`, `n_frames`, `n_tasks`, `n_trials` | how much data the session has |
| `duration_min` | recording length: first to last frame, using `pose_t` (camera time) |
| `task_time_min` | time actually spent on tasks: first `task_start` to last `task_end` |
| `has_meta` | `.meta.json` exists (not needed: `session_start` holds the same info) |
| `usable` | has frames **and** at least one trial |

- One session (`LSR`, 10/09) was **left recording overnight**: 15 h of recording for 40 min of tasks (1.77 M frames, 68% of all frames). This is why `task_time_min` is needed next to `duration_min`. The idle frames disappear later, when we keep only frames inside trials.

Nothing is removed here: the table only **flags** sessions. We decide what to drop afterwards.

In [3]:
# STEP 1: build the sessions table.
# For each session we read its events file and the time column of its frames file,
# then collect one summary row (who, when, how much data). All rows -> one DataFrame.

session_rows = []
for session_id in session_ids:
    events = read_events(session_id)
    # Only the camera time column is loaded (fast): enough to count frames and measure the recording length.
    frame_times = pd.read_csv(recordings_root / f"{session_id}.frames.csv", usecols=["pose_t"])["pose_t"]

    # Patient ID and dominant arm are typed by the operator during the session ("..._set" events).
    # Empty values are ignored. A session can have several (e.g. the ID was corrected).
    patient_ids = [e["patient_id"] for e in events if e["type"] == "patient_id_set" and e.get("patient_id")]
    arms = [e["dominant_arm"] for e in events if e["type"] == "dominant_arm_set" and e.get("dominant_arm")]

    # Which tasks were done, how many trials, and when the first task started / last task ended.
    task_ids = {e["task_id"] for e in events if e["type"] == "task_start"}
    n_trials = sum(e["type"] == "trial_end" for e in events)
    task_times = [e["wall_time"] for e in events if e["type"] in ("task_start", "task_end")]

    # If several IDs were typed, keep the last one (the final correction). "" = never entered.
    patient_id = patient_ids[-1] if patient_ids else ""

    # Date and start time come from the file name: upper_limb_<YYYYMMDD>_<HHMMSS>
    date, time = session_id.removeprefix("upper_limb_").split("_")

    session_rows.append({
        "session_id": session_id,
        "date": pd.to_datetime(date).date(),
        "start_time": f"{time[:2]}:{time[2:4]}",
        "patient_id": patient_id,
        "patient_ids_entered": ", ".join(dict.fromkeys(patient_ids)),  # all distinct IDs, in typing order
        "dominant_arm": ", ".join(dict.fromkeys(arms)),                 # e.g. "right" or "left, right"
        "looks_like_test": patient_id.lower().startswith("test"),       # test_33, tester, ...
        "n_events": len(events),
        "n_frames": len(frame_times),
        "n_tasks": len(task_ids),
        "n_trials": n_trials,
        # Recording length (first to last frame, camera time), in minutes. 0 if there are no frames.
        "duration_min": round((frame_times.max() - frame_times.min()) / 60, 1) if len(frame_times) else 0.0,
        # Time really spent on tasks (first task_start to last task_end), in minutes.
        "task_time_min": round((max(task_times) - min(task_times)) / 60, 1) if task_times else 0.0,
        "has_meta": (recordings_root / f"{session_id}.meta.json").exists(),
        # A session is usable if it has movement data AND at least one trial.
        "usable": len(frame_times) > 0 and n_trials > 0,
    })

sessions = pd.DataFrame(session_rows)
print(f"Sessions: {len(sessions)}")

Sessions: 112


In [4]:
# CHECK STEP 1: how many sessions fall in each category.
print(f"Total sessions:                 {len(sessions)}")
print(f"  empty events file:            {(sessions['n_events'] == 0).sum()}")
print(f"  no frames:                    {(sessions['n_frames'] == 0).sum()}")
print(f"  no trials:                    {(sessions['n_trials'] == 0).sum()}")
print(f"  usable (frames + trials):     {sessions['usable'].sum()}")
print()
print(f"Usable sessions with a patient ID:   {(sessions['usable'] & (sessions['patient_id'] != '')).sum()}")
print(f"  of which look like tests:          {(sessions['usable'] & sessions['looks_like_test']).sum()}")
print(f"Usable sessions without a patient ID: {(sessions['usable'] & (sessions['patient_id'] == '')).sum()}")
print(f"Sessions where the ID changed:        {sessions['patient_ids_entered'].str.contains(',').sum()}")

# One row per patient ID, usable sessions only: how many sessions, trials and minutes each patient has.
# Sessions without an ID are grouped under "(none)".
per_patient = (
    sessions[sessions["usable"]]
    .assign(patient_id=lambda d: d["patient_id"].replace("", "(none)"))
    .groupby("patient_id")
    .agg(sessions=("session_id", "size"), trials=("n_trials", "sum"),
         recorded_min=("duration_min", "sum"), task_min=("task_time_min", "sum"), first_date=("date", "min"), last_date=("date", "max"),
         looks_like_test=("looks_like_test", "first"))
    .sort_values("trials", ascending=False)
)
per_patient

Total sessions:                 112
  empty events file:            4
  no frames:                    6
  no trials:                    33
  usable (frames + trials):     79

Usable sessions with a patient ID:   29
  of which look like tests:          20
Usable sessions without a patient ID: 50
Sessions where the ID changed:        1


,sessions,trials,recorded_min,task_min,first_date,last_date,looks_like_test
patient_id,,,,,,,
(none),50,1339,205.7,156.4,2026-06-29,2026-07-29,False
DBB,1,289,37.6,36.9,2026-08-10,2026-08-10,False
MBM,1,270,31.1,30.7,2026-08-11,2026-08-11,False
LSR,1,265,900.3,36.9,2026-09-10,2026-09-10,False
004,1,265,35.1,33.8,2026-09-14,2026-09-14,False
ISA,1,156,28.9,21.6,2026-09-08,2026-09-08,False
FCG,1,151,25.4,23.8,2026-09-03,2026-09-03,False
anon,2,117,26.1,25.4,2026-07-29,2026-07-30,False
MFC,1,96,14.5,13.6,2026-09-02,2026-09-02,False


In [5]:
# Show the full sessions table (all rows and columns, nothing hidden behind "...").
# with pd.option_context("display.max_rows", None, "display.max_columns", None):
#     display(sessions)

## Step 2: trials table (one row per trial = one movement)

A trial is everything between a `trial_start` and the next `trial_end` event of a session. What we checked in the data:

- Every `trial_start` has exactly one matching `trial_end` (no missing or overlapping trials).
- **A task can be run several times in the same session** (73 cases), and `trial_index` restarts at 1 each time. So `task_id` + `trial_index` is not unique. We add `task_run` (1st, 2nd, ... time the task was started in the session) and give each trial a unique `trial_id`.
- **Why a trial ended** (`end_reason`) falls into 4 groups (`end_type`):

| `end_type` | `end_reason` values | meaning |
|---|---|---|
| `auto` | `start_returned` | patient brought the arm back to start: normal, complete trial |
| `manual` | `manual_*` | operator pressed a key/button to go to the next trial |
| `interrupted` | `shutdown`, `replaced`, `operator_stop` | session stopped or trial restarted |
| `failed` | `returned_to_a_without_target` | patient came back without reaching the target |

- `goal_reached` is True when the trial contains a "goal" event (`target_reached`, `circle_completed`, `cup_reached`, `mouth_reached` or `nose_reached`).
- The frames of a trial are those with `wall_time` between `start_time` and `end_time`. This gives exactly the same frames as the flags in the frames file (`trial_active == 1` + `active_task_id` + `trial_index`); we checked all 3013 trials.

Columns:

| column | meaning |
|---|---|
| `trial_id` | unique ID: `<session_id>_<number of the trial in the session>`, e.g. `upper_limb_20260629_095022_007` |
| `session_id` | link to the sessions table |
| `task_id`, `task_name`, `task_run` | which task, and which run of that task in the session |
| `trial_index`, `trial_count` | trial number within the task run, out of how many |
| `side` | arm used: `left` / `right` |
| `start_time`, `pose_locked_time`, `end_time` | `wall_time` of `trial_start`, `trial_start_pose_locked` (patient in start pose, empty if never) and `trial_end` |
| `duration_s` | from the `trial_end` event |
| `end_reason`, `end_type`, `goal_reached` | how the trial ended (see above) |
| `n_frames` | number of frames inside the trial |

Again, nothing is removed: these columns let you choose trials later.

In [6]:
# STEP 2: build the trials table.
# For each session we walk through its events in time order. A trial starts at "trial_start"
# and ends at the next "trial_end"; everything in between (start pose, target reached, ...)
# is noted on that trial. When the trial ends, it becomes one row of the table.

import numpy as np

# Events that mean "the patient did what the task asked" (the goal differs per task).
GOAL_EVENTS = {"target_reached", "circle_completed", "cup_reached", "mouth_reached", "nose_reached"}


def end_type(reason):
    """Group the ~12 different end reasons into 4 simple categories."""
    if reason == "start_returned":
        return "auto"         # patient came back to the start position: normal end
    if reason.startswith("manual"):
        return "manual"       # operator pressed a key/button to go on
    if reason == "returned_to_a_without_target":
        return "failed"       # patient came back without reaching the target
    return "interrupted"      # shutdown, replaced, operator_stop


trial_rows = []
for session_id in session_ids:
    events = read_events(session_id)
    if not events:  # empty events file -> no trials
        continue
    # Times of all frames of the session (sorted), used below to count the frames inside each trial.
    frame_times = np.sort(pd.read_csv(recordings_root / f"{session_id}.frames.csv", usecols=["wall_time"])["wall_time"].to_numpy())

    task_runs = {}   # how many times each task has been started so far in this session
    current = None   # the trial being filled in (None = we are between two trials)
    for event in events:
        event_type = event["type"]

        if event_type == "task_start":
            # A task starts (again): count it, so repeated runs of the same task get task_run = 1, 2, ...
            task_runs[event["task_id"]] = task_runs.get(event["task_id"], 0) + 1

        elif event_type == "trial_start":
            # A new trial begins: open a new row with what we already know.
            current = {
                "session_id": session_id,
                "task_id": event["task_id"],
                "task_name": event["task_name"],
                "task_run": task_runs.get(event["task_id"], 1),
                "trial_index": event["trial_index"],
                "trial_count": event["trial_count"],
                "side": event["side"],
                "start_time": event["wall_time"],
                "pose_locked_time": np.nan,   # filled in below if the patient reaches the start pose
                "goal_reached": False,        # set to True below if a goal event happens
            }

        elif current is None:
            # Event between two trials (countdown, task_start, operator keys, ...): not part of a trial.
            continue

        elif event_type == "trial_start_pose_locked":
            # Patient is in the start position: the movement can begin.
            current["pose_locked_time"] = event["wall_time"]

        elif event_type in GOAL_EVENTS:
            current["goal_reached"] = True

        elif event_type == "trial_end":
            # The trial is over: complete the row and store it.
            current["end_time"] = event["wall_time"]
            current["duration_s"] = event["duration_s"]
            current["end_reason"] = event["reason"]
            current["end_type"] = end_type(event["reason"])
            # Number of frames with start_time <= wall_time <= end_time.
            # searchsorted finds where each time would fit in the sorted list; the difference is the count.
            current["n_frames"] = int(
                np.searchsorted(frame_times, current["end_time"], side="right")
                - np.searchsorted(frame_times, current["start_time"], side="left")
            )
            trial_rows.append(current)
            current = None

trials = pd.DataFrame(trial_rows)

# Unique trial ID = session ID + position of the trial in the session (001, 002, ...).
trials.insert(0, "trial_id", trials["session_id"] + "_" + (trials.groupby("session_id").cumcount() + 1).map("{:03d}".format))

# Put the columns in a readable order.
trials = trials[[
    "trial_id", "session_id", "task_id", "task_name", "task_run", "trial_index", "trial_count", "side",
    "start_time", "pose_locked_time", "end_time", "duration_s", "end_reason", "end_type", "goal_reached", "n_frames",
]]

assert trials["trial_id"].is_unique  # safety check: no two trials share an ID
print(f"Trials: {len(trials)} in {trials['session_id'].nunique()} sessions")

Trials: 3013 in 79 sessions


In [7]:
# CHECK STEP 2: how did the trials end, and how many trials per task and arm?

# Table: rows = how the trial ended, columns = goal reached or not, with totals.
print("How trials ended (rows) and whether the goal was reached (columns):")
display(pd.crosstab(trials["end_type"], trials["goal_reached"], margins=True, margins_name="total"))

# Special cases worth knowing about.
print(f"Trials with no frames:          {(trials['n_frames'] == 0).sum()}")
print(f"Trials never locked in start pose: {trials['pose_locked_time'].isna().sum()}")

# Spread of trial durations: min, 5% shortest, median, 95%, max.
print("\nTrial duration (s):")
print(trials["duration_s"].describe(percentiles=[0.05, 0.5, 0.95]).round(1).to_string())

# One row per task: number of trials per arm, % of trials where the goal was reached, typical duration.
per_task = trials.pivot_table(index=["task_id", "task_name"], columns="side", values="trial_id", aggfunc="count", fill_value=0)
per_task["total"] = per_task.sum(axis=1)
per_task["goal_reached_%"] = (trials.groupby(["task_id", "task_name"])["goal_reached"].mean() * 100).round(0)
per_task["median_duration_s"] = trials.groupby(["task_id", "task_name"])["duration_s"].median().round(1)
per_task

How trials ended (rows) and whether the goal was reached (columns):


goal_reached,False,True,total
end_type,,,
auto,7,1783,1790
failed,3,0,3
interrupted,55,15,70
manual,123,1027,1150
total,188,2825,3013


Trials with no frames:          2
Trials never locked in start pose: 38

Trial duration (s):
count    3013.0
mean        6.4
std        12.8
min         0.0
5%          1.5
50%         3.6
95%        19.3
max       454.1


,side,left,right,total,goal_reached_%,median_duration_s
task_id,task_name,,,,,
1,Shoulder flexion with elbow extended,120,354,474,85.0,3.4
2,Shoulder abduction with elbow extended,117,167,284,96.0,3.1
3,Draw a 5 segment trajectory,93,130,223,94.0,14.7
4,Reaching different targets,208,337,545,97.0,2.9
5,Draw a circle,86,131,217,93.0,14.6
6,Horizontal reaching,252,472,724,98.0,3.2
7,Drinking test: reaching glass to mouth,125,153,278,92.0,6.5
8,Index to nose,120,148,268,91.0,2.4


In [8]:
# First 10 rows of the trials table, to see what it looks like.
trials.head(10)

,trial_id,session_id,task_id,task_name,task_run,trial_index,trial_count,side,start_time,pose_locked_time,end_time,duration_s,end_reason,end_type,goal_reached,n_frames
0,upper_limb_20260629_090253_001,upper_limb_20260629_090253,1,Shoulder flexion with elbow extended,1,1,10,right,1.782720e+09,1.782720e+09,1.782720e+09,7.013817,start_returned,auto,True,123
1,upper_limb_20260629_090253_002,upper_limb_20260629_090253,1,Shoulder flexion with elbow extended,1,2,10,right,1.782720e+09,1.782720e+09,1.782720e+09,1.452592,start_returned,auto,True,25
2,upper_limb_20260629_090253_003,upper_limb_20260629_090253,1,Shoulder flexion with elbow extended,1,3,10,right,1.782720e+09,1.782720e+09,1.782720e+09,1.393823,start_returned,auto,True,24
3,upper_limb_20260629_090253_004,upper_limb_20260629_090253,1,Shoulder flexion with elbow extended,1,4,10,right,1.782720e+09,1.782720e+09,1.782720e+09,1.458956,start_returned,auto,True,26
4,upper_limb_20260629_090253_005,upper_limb_20260629_090253,1,Shoulder flexion with elbow extended,1,5,10,right,1.782720e+09,1.782720e+09,1.782720e+09,1.388258,start_returned,auto,True,24
5,upper_limb_20260629_090253_006,upper_limb_20260629_090253,1,Shoulder flexion with elbow extended,1,6,10,right,1.782720e+09,1.782720e+09,1.782720e+09,1.316887,start_returned,auto,True,23
6,upper_limb_20260629_090253_007,upper_limb_20260629_090253,1,Shoulder flexion with elbow extended,1,7,10,right,1.782720e+09,1.782720e+09,1.782720e+09,1.364561,start_returned,auto,True,23
7,upper_limb_20260629_090253_008,upper_limb_20260629_090253,1,Shoulder flexion with elbow extended,1,8,10,right,1.782720e+09,1.782720e+09,1.782720e+09,1.295537,start_returned,auto,True,21
8,upper_limb_20260629_090253_009,upper_limb_20260629_090253,1,Shoulder flexion with elbow extended,1,9,10,right,1.782720e+09,1.782720e+09,1.782720e+09,1.361945,start_returned,auto,True,23
9,upper_limb_20260629_090253_010,upper_limb_20260629_090253,1,Shoulder flexion with elbow extended,1,10,10,right,1.782720e+09,1.782720e+09,1.782720e+09,1.459821,start_returned,auto,True,25


## Save the tables

Until now the tables only exist in memory (they disappear when the notebook is closed). This step writes them to `data/processed/` as parquet files:

| file | content |
|---|---|
| `sessions.parquet` | step 1: one row per session |
| `trials.parquet` | step 2: one row per trial |

The raw recordings are not touched. `data/` is in `.gitignore`, so these files are never committed to git.

In [9]:
# SAVE: write the tables built so far to data/processed/.
# Parquet is a compact table format: fast to load, and keeps column types (dates, True/False, numbers).
# Load them back anywhere with: pd.read_parquet(processed_root / "sessions.parquet")
# Running this cell again simply overwrites the files with the new version.

processed_root.mkdir(parents=True, exist_ok=True)  # create the folder if it does not exist yet

sessions.to_parquet(processed_root / "sessions.parquet", index=False)
trials.to_parquet(processed_root / "trials.parquet", index=False)

# Check: list what is in the folder, with the size of each file.
for path in sorted(processed_root.iterdir()):
    print(f"{path.name:<20} {path.stat().st_size / 1024:8.1f} KB")

sessions.parquet         12.4 KB
trials.parquet          132.7 KB
